In [0]:
inventory = spark.table(
    "workspace.pharma_silver.silver_inventory"
)

drugs = spark.table(
    "workspace.pharma_silver.silver_drugs"
)

suppliers = spark.table(
    "workspace.pharma_silver.silver_suppliers"
)

In [0]:
df_inventory_gold = (
    inventory
    .join(
        drugs,
        on="drug_id",
        how="left"
    )
    .join(
        suppliers,
        on="supplier_id",
        how="left"
    )
)

In [0]:
from pyspark.sql.functions import col, round, when

df_inventory_gold = (
    df_inventory_gold
    .withColumn(
        "inventory_value",
        round(
            col("quantity_on_hand") * col("unit_cost"),
            2
        )
    )
)

In [0]:
df_inventory_gold = df_inventory_gold.withColumn(
    "reorder_required",
    when(
        col("quantity_on_hand") <= col("reorder_level"),
        "YES"
    ).otherwise("NO")
)

In [0]:
df_inventory_gold = df_inventory_gold.withColumn(
    "stock_status",
    when(
        col("quantity_on_hand") <= col("reorder_level"),
        "LOW STOCK"
    ).otherwise("SUFFICIENT STOCK")
)

In [0]:
df_inventory_gold = df_inventory_gold.select(
    "inventory_id",
    "drug_id",
    "drug_name",
    "supplier_id",
    "supplier_name",
    "warehouse_location",
    "quantity_on_hand",
    "reorder_level",
    "unit_cost",
    "inventory_value",
    "reorder_required",
    "stock_status"
)

In [0]:
display(df_inventory_gold)

In [0]:
df_inventory_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pharma_gold.gold_inventory_analytics")